# Prepare training set

In [5]:
import os
import subprocess
import glob
import re


In [6]:
raw_data="../../data/species"
lyric_data="../../lyric_annotator"
isoquant_data="../../isoquant_annotator"

!ls $raw_data

Acanthamoeba_polyphaga_5757		   Micromonas_pusilla_CCMP1545_38833
Babesia_duncani_323732			   Naegleria_gruberi_5762
Caulerpa_lentillifera_148947		   Paramecium_bursaria_74790
Chaetoceros_neogracilis_240364		   Paramecium_tetraurelia_5888
Chlamydomonas_reinhardtii_3055		   Phytophthora_agathidicida_1642459
Chlorella_sorokiniana_3076		   Plasmodium_berghei_ANKA_5821
Cladocopium_goreaui_2562237		   Plasmodium_falciparum_3D7_36329
Conticribra_weissflogii_1577725		   Plasmodium_ovale_36330
Cryptosporidium_parvum_Iowa_II_5807	   Plasmodium_vivax_5855
Cyanidiococcus_yangmingshanensis_2690220   Plasmodium_yoelii_yoelii_73239
Cyanidioschyzon_merolae_strain_10D_280699  Pseudo-nitzschia_delicatissima_44447
Cyanidium_caldarium_2771		   Pseudo-nitzschia_multiseries_37319
Cylindrotheca_closterium_2856		   Pseudo-nitzschia_pungens_37318
Dunaliella_salina_3046			   Pyrocystis_lunula_2972
Eimeria_necatrix_51315			   Pyropia_haitanensis_1262161
Entamoeba_histolytica_HM-1:IMSS_5759	   Pyropia_yezoensis_

In [ ]:
#annotation sources, species can appear in multiple to compare
reference_all=!ls $raw_data
lyric_all=reference_all
isoquant_all=reference_all

#filter out
#banned_species=["Cladocopium_goreaui", "Schizochytrium_sp._CCTCC_M209059", "Plasmodium_ovale", "Pseudo-nitzschia_multiseries", "Tetrahymena_thermophila_SB210", "Pyrocystis_lunula", "Micractinium_conductrix_554055"]
banned_species=[]

reference=[sp for sp in reference_all if re.sub(r'_\d+$', '', sp) not in banned_species]
lyric=[sp for sp in lyric_all if re.sub(r'_\d+$', '', sp) not in banned_species]
isoquant=[sp for sp in isoquant_all if re.sub(r'_\d+$', '', sp) not in banned_species]

print(f"Reference: {len(reference)}/{len(reference_all)} species")
print(f"Lyric: {len(lyric)}/{len(lyric_all)} species")
print(f"Isoquant: {len(isoquant)}/{len(isoquant_all)} species")

baseAnn_origins={"reference":reference, "lyric":lyric, "isoquant":isoquant}

#ordered glob templates per source, resolve for GCA and GCF
src_glob={
    "reference": [raw_data+"/{sp}/GC*/{sp}_*_GC*.gff.gz"],
    "lyric":     [lyric_data+"/summary/merge/pred/{sp}_mergedRef.gff",
                  lyric_data+"/summary/pred/{sp}_pred.gff"],
    "isoquant":  [isoquant_data+"/summary/pred/{sp}_pred.gff"],
}

#composite work-unit key: <species>_<source>. Source names must contain no "_" so that key.rsplit("_",1) recovers (species, source).
def unit_key(sp, src):
    return f"{sp}_{src}"

def resolve_path(sp, src):
    """First src_glob pattern for (sp, src) that resolves to exactly one file, else None."""
    tried=[]
    for pat in src_glob[src]:
        pattern=pat.format(sp=sp)
        tried.append(pattern)
        matches=glob.glob(pattern)
        if len(matches)==1:
            return matches[0]
        if len(matches)>1:
            print(f"Ambiguous {src} annotation for {sp}:\n{matches}")
            return None
    print(f"No {src} annotation for {sp} (tried: {tried})")
    return None

def resolve_units():
    """(sp, src, ann_path) for every assigned species with a resolvable annotation."""
    units=[]
    for src, names in baseAnn_origins.items():
        for sp in names:
            path=resolve_path(sp, src)
            if path:
                units.append((sp, src, path))
    return units

print(baseAnn_origins)

Reference: 48/53 species
Lyric: 48/53 species
Isoquant: 48/53 species
{'reference': ['Acanthamoeba_polyphaga_5757', 'Babesia_duncani_323732', 'Caulerpa_lentillifera_148947', 'Chaetoceros_neogracilis_240364', 'Chlamydomonas_reinhardtii_3055', 'Chlorella_sorokiniana_3076', 'Conticribra_weissflogii_1577725', 'Cryptosporidium_parvum_Iowa_II_5807', 'Cyanidiococcus_yangmingshanensis_2690220', 'Cyanidioschyzon_merolae_strain_10D_280699', 'Cyanidium_caldarium_2771', 'Cylindrotheca_closterium_2856', 'Dunaliella_salina_3046', 'Eimeria_necatrix_51315', 'Entamoeba_histolytica_HM-1:IMSS_5759', 'Euglena_gracilis_3039', 'Galdieria_yellowstonensis_3028027', 'Giardia_duodenalis_5741', 'Giardia_muris_5742', 'Haematococcus_lacustris_44745', 'Hydrurus_foetidus_2996', 'Isochrysis_galbana_37099', 'Lagenidium_giganteum_4803', 'Leishmania_donovani_5661', 'Leishmania_infantum_JPCM5_5671', 'Micractinium_conductrix_554055', 'Micromonas_pusilla_CCMP1545_38833', 'Naegleria_gruberi_5762', 'Paramecium_bursaria_74790

## Clean assemblies

In [9]:
#genome is shared across a species' sources, so clean each assembly once
tr_sp=sorted({sp for sp, _, _ in resolve_units()})

for sp in tr_sp:

    ref_assembly=!realpath $raw_data/$sp/GC[AF]*/*_genomic*.fna*
    ref_assembly=ref_assembly[0]
    ref_assembly_name=ref_assembly.split("/")[-1].replace(".gz", "")
    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/CLEAN_{ref_assembly_name}"
    print(result_file)
    
    !bash ../scripts/clean_ref.sh $ref_assembly > $result_file

No reference annotation for Acanthamoeba_polyphaga_5757 (tried: ['../../data/species/Acanthamoeba_polyphaga_5757/GC*/Acanthamoeba_polyphaga_5757_*_GC*.gff.gz'])
No reference annotation for Caulerpa_lentillifera_148947 (tried: ['../../data/species/Caulerpa_lentillifera_148947/GC*/Caulerpa_lentillifera_148947_*_GC*.gff.gz'])
No reference annotation for Chlorella_sorokiniana_3076 (tried: ['../../data/species/Chlorella_sorokiniana_3076/GC*/Chlorella_sorokiniana_3076_*_GC*.gff.gz'])
No reference annotation for Cryptosporidium_parvum_Iowa_II_5807 (tried: ['../../data/species/Cryptosporidium_parvum_Iowa_II_5807/GC*/Cryptosporidium_parvum_Iowa_II_5807_*_GC*.gff.gz'])
No reference annotation for Entamoeba_histolytica_HM-1:IMSS_5759 (tried: ['../../data/species/Entamoeba_histolytica_HM-1:IMSS_5759/GC*/Entamoeba_histolytica_HM-1:IMSS_5759_*_GC*.gff.gz'])
No reference annotation for Euglena_gracilis_3039 (tried: ['../../data/species/Euglena_gracilis_3039/GC*/Euglena_gracilis_3039_*_GC*.gff.gz'])
N

## Extract CDS from annotations

In [10]:
#one CDS extraction per (species, source) unit; source-tagged names coexist
for sp, src, ref_ann in resolve_units():
    print("##########################################################")
    print(f"{sp} -> {src}: {ref_ann}")

    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/CDS_{unit_key(sp, src)}.gff"
    print(result_file)

    !bash ../scripts/get_CDS.sh $ref_ann > $result_file

!find ../training_data/species/ -path "*/CDS*" -type f -size 0 -print
!find ../training_data/species/ -path "*/CDS*" -type f -size 0 -delete
!ls -lh ../training_data/species/**/CDS*

No reference annotation for Acanthamoeba_polyphaga_5757 (tried: ['../../data/species/Acanthamoeba_polyphaga_5757/GC*/Acanthamoeba_polyphaga_5757_*_GC*.gff.gz'])
No reference annotation for Caulerpa_lentillifera_148947 (tried: ['../../data/species/Caulerpa_lentillifera_148947/GC*/Caulerpa_lentillifera_148947_*_GC*.gff.gz'])
No reference annotation for Chlorella_sorokiniana_3076 (tried: ['../../data/species/Chlorella_sorokiniana_3076/GC*/Chlorella_sorokiniana_3076_*_GC*.gff.gz'])
No reference annotation for Cryptosporidium_parvum_Iowa_II_5807 (tried: ['../../data/species/Cryptosporidium_parvum_Iowa_II_5807/GC*/Cryptosporidium_parvum_Iowa_II_5807_*_GC*.gff.gz'])
No reference annotation for Entamoeba_histolytica_HM-1:IMSS_5759 (tried: ['../../data/species/Entamoeba_histolytica_HM-1:IMSS_5759/GC*/Entamoeba_histolytica_HM-1:IMSS_5759_*_GC*.gff.gz'])
No reference annotation for Euglena_gracilis_3039 (tried: ['../../data/species/Euglena_gracilis_3039/GC*/Euglena_gracilis_3039_*_GC*.gff.gz'])
N

## Sample n genes from CDS annotations

In [11]:
n=1000
#sample each source's CDS file independently (one per unit)
for cds_path in sorted(glob.glob("../training_data/species/*/CDS_*.gff")):
    cds_name=os.path.basename(cds_path)
    sp=os.path.basename(os.path.dirname(cds_path))
    print(cds_name)
    #create folder if needed
    result_folder=f"../training_data/species/{sp}"
    os.makedirs(result_folder, exist_ok=True)
    result_file=f"{result_folder}/sample{n}_{cds_name}"
    print(result_file)

    !bash ../scripts/sample_CDS.sh $cds_path $n > $result_file

CDS_Acanthamoeba_polyphaga_5757_lyric.gff
../training_data/species/Acanthamoeba_polyphaga_5757/sample1000_CDS_Acanthamoeba_polyphaga_5757_lyric.gff
CDS_Babesia_duncani_323732_isoquant.gff
../training_data/species/Babesia_duncani_323732/sample1000_CDS_Babesia_duncani_323732_isoquant.gff
CDS_Babesia_duncani_323732_lyric.gff
../training_data/species/Babesia_duncani_323732/sample1000_CDS_Babesia_duncani_323732_lyric.gff
CDS_Babesia_duncani_323732_reference.gff
../training_data/species/Babesia_duncani_323732/sample1000_CDS_Babesia_duncani_323732_reference.gff
CDS_Caulerpa_lentillifera_148947_lyric.gff
../training_data/species/Caulerpa_lentillifera_148947/sample1000_CDS_Caulerpa_lentillifera_148947_lyric.gff
CDS_Chaetoceros_neogracilis_240364_isoquant.gff
../training_data/species/Chaetoceros_neogracilis_240364/sample1000_CDS_Chaetoceros_neogracilis_240364_isoquant.gff
CDS_Chaetoceros_neogracilis_240364_lyric.gff
../training_data/species/Chaetoceros_neogracilis_240364/sample1000_CDS_Chaetocer

# Training commands

In [12]:
n=1000

!mkdir -p ../job_commands
with open("../job_commands/total_training.txt", "w") as out:
    
    #create trained parameters folder command
    parameters_folder="../results/trainedParams"
    print(f"mkdir -p {parameters_folder}", file=out)
    #one training block per (species, source) unit, keyed off its sampled CDS file
    for sampleN_CDS in sorted(glob.glob(f"../training_data/species/*/sample{n}_CDS_*.gff")):
        sampleN_CDS=os.path.abspath(sampleN_CDS)
        #unit = <species>_<source>, recovered from the CDS filename
        unit=os.path.basename(sampleN_CDS).replace(f"sample{n}_CDS_", "").replace(".gff", "")
        sp=unit.rsplit("_", 1)[0]

        #shared cleaned genome for the species
        clean_ref=glob.glob(f"../training_data/species/{sp}/CLEAN_*.fna")
        if not clean_ref:
            print(f"No cleaned genome for {sp}")
            continue
        clean_ref=os.path.abspath(clean_ref[0])

        #set folder for results
        result_sp=f"../results/specie_logs/{unit}/"

        #training command using singularity container
        geneidtrainer_command=f"singularity run \
                            ~/software/geneidtrainerdocker.sif \
                            /scripts_geneid/geneidTRAINer4docker.pl \
                            -species {unit} \
                            -gff {sampleN_CDS} \
                            -fastas {clean_ref} \
                            -results {result_sp} \
                            -reduced no"
        geneidtrainer_command=geneidtrainer_command.replace("                             ", " ")

        #copy parameter files for clear access(+git)
        #move them to folder that will be in git, and link them to unit specific
        mvParam_command=f"mv {result_sp}*.optimized.param {parameters_folder}/"
        lnParam_command=f"ln -vf {parameters_folder}/{unit}.geneid.optimized.param {result_sp}"


        print(f'echo "Specie: {unit}"', file=out)
        print(geneidtrainer_command)
        print(geneidtrainer_command, file=out)
        print(mvParam_command, file=out)
        print(lnParam_command, file=out)

subprocess.run(["cp", "../scripts/memory/trainer.sh", "../job_commands/"])
subprocess.run(["bash", "../scripts/missing_train.sh"])

#Execute generated commands to train geneId

singularity run ~/software/geneidtrainerdocker.sif /scripts_geneid/geneidTRAINer4docker.pl -species Acanthamoeba_polyphaga_5757_lyric -gff /users/rg/jizquierdo/git/geneid-training/training_data/species/Acanthamoeba_polyphaga_5757/sample1000_CDS_Acanthamoeba_polyphaga_5757_lyric.gff -fastas /users/rg/jizquierdo/git/geneid-training/training_data/species/Acanthamoeba_polyphaga_5757/CLEAN_GCA_001567625.1_ASM156762v1_genomic.fna -results ../results/specie_logs/Acanthamoeba_polyphaga_5757_lyric/ -reduced no
singularity run ~/software/geneidtrainerdocker.sif /scripts_geneid/geneidTRAINer4docker.pl -species Babesia_duncani_323732_isoquant -gff /users/rg/jizquierdo/git/geneid-training/training_data/species/Babesia_duncani_323732/sample1000_CDS_Babesia_duncani_323732_isoquant.gff -fastas /users/rg/jizquierdo/git/geneid-training/training_data/species/Babesia_duncani_323732/CLEAN_GCF_028658345.1_Bduncani_v1_genomic.fna -results ../results/specie_logs/Babesia_duncani_323732_isoquant/ -reduced no
si

CompletedProcess(args=['bash', '../scripts/missing_train.sh'], returncode=0)